<div align="center">

  <h1>📖 Fable Demo</h1>

  <h3>Generate and fine-tune a compact storytelling language model</h3>

</div>

---


This notebook mirrors the quickstart workflow from the README.

It covers:
- installing the library on Colab;
- loading the pretrained demo checkpoint;
- generating stories at different temperatures;
- (optional) preparing TinyStories data and launching a brief training run.

Feel free to skip any sections you don't need.


## 1. Environment Setup

Install the latest Fable build directly from GitHub. Re-run this cell after enabling a GPU runtime in Colab.


In [ ]:
!pip install --quiet git+https://github.com/auxeno/fable

### Check available devices

JAX automatically selects GPU/TPU devices when available.


In [ ]:
import jax
jax.devices()

## 2. Load the demo model

This pulls the bundled `demo.ckpt` checkpoint into memory.


In [ ]:
from fable.checkpoint import load
from fable.generate import generate_text

demo_model = load()
demo_model.config

## 3. Generate a short story

Use the helper below to sample text without streaming delays.


In [ ]:
import contextlib
import io

def sample_story(prompt: str, *, temperature: float = 0.6, max_tokens: int = 256, seed: int = 0):
    """Generate text with fast console output."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        generate_text(
            prompt,
            model=demo_model,
            temperature=temperature,
            max_output_tokens=max_tokens,
            chars_per_second=10_000,
            seed=seed,
        )
    story = buffer.getvalue().strip()
    print(story)

In [ ]:
sample_story("Lily got a new puppy", temperature=0.6)

### Temperature comparison

Lower values stay close to the training data, higher values explore creative continuations.


In [ ]:
for temp in (0.4, 0.6, 0.8):
    print(f"\n--- temperature {temp} ---")
    sample_story("Lily got a new puppy", temperature=temp, seed=42)

## 4. (Optional) Prepare TinyStories data

Run the full download → clean → tokenise pipeline. This writes data into the `data/` directory.

> ⚠️ This step downloads ~250 MB and can take a few minutes.


In [ ]:
from fable.data import prepare_tinystories_dataset

prepare_tinystories_dataset()

## 5. (Optional) Quick training run

This example trains a smaller model variant for a single epoch. Increase `num_epochs` for longer runs.

> 💡 Ensure the TinyStories data pipeline has run before executing this cell.


In [ ]:
import jax
from flax import nnx
from fable.config import GPTConfig
from fable.model import GPT
from fable.train import train

small_config = GPTConfig(
    num_layers=2,
    embed_dim=128,
    num_heads=4,
    max_seq_len=128,
    batch_size=32,
    num_epochs=1,
    enable_checkpointing=False,
    verbose=True,
)

rng = jax.random.PRNGKey(small_config.seed)
scratch_model = GPT(config=small_config, rngs=nnx.Rngs(rng))
trained_model = train(model=scratch_model)

---

You're ready to explore the rest of the project! Check out the CLI tools (`fable-generate`, `fable-train`) and the README for more details.
